In [22]:
from torch_geometric.datasets import Planetoid, Coauthor, CitationFull, WikipediaNetwork, TUDataset, ZINC, QM9
from torch_geometric.data import DataLoader
import torch
from utils import coarsening_classification, coarsening_regression, load_graph_data
import argparse
from tqdm import tqdm

In [70]:
def process_dataset(args):
    # Node Classification
    if args.dataset == 'dblp':
        dataset = CitationFull(root='./dataset', name=args.dataset)
        if args.normalize_features:
            dataset.x = torch.nn.functional.normalize(dataset.x, p=1)
        args.task = 'node_cls'
    elif args.dataset == 'Physics':
        dataset = Coauthor(root='./dataset/Physics', name=args.dataset)
        if args.normalize_features:
            dataset.x = torch.nn.functional.normalize(dataset.x, p=1)
        args.task = 'node_cls'
    elif args.dataset == 'cora':
        dataset = Planetoid(root='./dataset', name=args.dataset)
        if args.normalize_features:
            dataset.x = torch.nn.functional.normalize(dataset.x, p=1)
        args.task = 'node_cls'
    elif args.dataset == 'citeseer':
        dataset = Planetoid(root='./dataset', name=args.dataset)
        if args.normalize_features:
            dataset.x = torch.nn.functional.normalize(dataset.x, p=1)
        args.task = 'node_cls'
    elif args.dataset == 'pubmed':
        dataset = Planetoid(root='./dataset', name=args.dataset)
        if args.normalize_features:
            dataset.x = torch.nn.functional.normalize(dataset.x, p=1)
        args.task = 'node_cls'
    #Node Regression
    elif args.dataset == 'chameleon':
        dataset = WikipediaNetwork(root='./dataset', name=args.dataset, geom_gcn_preprocess=False)
        if args.normalize_features:
            dataset.x = torch.nn.functional.normalize(dataset.x, p=1)
        args.task = 'node_reg'
    elif args.dataset == 'squirrel':
        dataset = WikipediaNetwork(root='./dataset', name=args.dataset, geom_gcn_preprocess=False)
        if args.normalize_features:
            dataset.x = torch.nn.functional.normalize(dataset.x, p=1)
        args.task = 'node_reg'
    elif args.dataset == 'crocodile':
        dataset = WikipediaNetwork(root='./dataset', name=args.dataset, geom_gcn_preprocess=False)
        if args.normalize_features:
            dataset.x = torch.nn.functional.normalize(dataset.x, p=1)
        args.task = 'node_reg'
    #Graph Classification
    elif args.dataset == 'ENZYMES':
        dataset = TUDataset(root='./dataset', name=args.dataset)
        if args.normalize_features:
            for i in range(len(dataset)):
                dataset[i].x = torch.nn.functional.normalize(dataset[i].x, p=1)
        args.task = 'graph_cls'
    elif args.dataset == 'PROTEINS':
        dataset = TUDataset(root='./dataset', name=args.dataset)
        if args.normalize_features:
            for i in range(len(dataset)):
                dataset[i].x = torch.nn.functional.normalize(dataset[i].x, p=1)
        args.task = 'graph_cls'
    elif args.dataset == 'AIDS':
        dataset = TUDataset(root='./dataset', name=args.dataset)
        args.task = 'graph_cls'
    #Graph Regression
    elif args.dataset == 'QM9':
        dataset = QM9(root='./dataset/QM9')
        args.task = 'graph_reg'
        args.multi_prop = True
    elif args.dataset == 'ZINC_full':
        dataset = ZINC(root='./dataset/ZINC', subset=False)
        args.task = 'graph_reg'
    elif args.dataset == 'ZINC_subset':
        dataset = ZINC(root='./dataset/ZINC', subset=True)
        args.task = 'graph_reg'
    return dataset, args

def arg_correction(args):
    if args.super_graph:
        args.cluster_node = False
        args.extra_node = False
    elif args.cluster_node:
        args.extra_node = False
        args.super_graph = False
    elif args.extra_node:
        args.cluster_node = False
        args.super_graph = False
    return args

parser = argparse.ArgumentParser()
parser.add_argument('--dataset', type=str, default='cora')
parser.add_argument('--experiment', type=str, default='fixed')
parser.add_argument('--runs', type=int, default=20)
parser.add_argument('--exp_setup', type=str, default='Gc_train_2_Gs_infer')
parser.add_argument('--hidden', type=int, default=512)
parser.add_argument('--epochs1', type=int, default=100)
parser.add_argument('--epochs2', type=int, default=300)
parser.add_argument('--num_layers1', type=int, default=2)
parser.add_argument('--num_layers2', type=int, default=2)
parser.add_argument('--batch_size', type=int, default=128)
parser.add_argument('--train_ratio', type=float, default=0.3)
parser.add_argument('--val_ratio', type=float, default=0.2)
parser.add_argument('--early_stopping', type=int, default=10)
parser.add_argument('--extra_node', type=bool, default=False)
parser.add_argument('--cluster_node', type=bool, default=False)
parser.add_argument('--super_graph', type=bool, default=False)
parser.add_argument('--lr', type=float, default=0.01)
parser.add_argument('--weight_decay', type=float, default=0.0005)
parser.add_argument('--normalize_features', type=bool, default=True)
parser.add_argument('--coarsening_ratio', type=float, default=0.1)
parser.add_argument('--coarsening_method', type=str, default='variation_neighborhoods')
# parser.add_argument('--output_dir', type=str, required=True)
parser.add_argument('--task', type = str, default = 'node_cls')
parser.add_argument('--seed', type = int, default = None)
parser.add_argument('--multi_prop', type =bool, default=False)
parser.add_argument('--property', type = int, default = 0)
args = parser.parse_args([])

args = arg_correction(args)
dataset, args = process_dataset(args)

In [25]:
def save_graph(dataset, args, path_to_save):
    if args.task == 'node_cls':
        args.num_features, candidate, C_list, Gc_list, subgraph_list, component_2_subgraphs, CLIST, GcLIST = coarsening_classification(args, dataset[0], 1-args.coarsening_ratio, args.coarsening_method)
        return args.num_features, candidate, C_list, Gc_list, subgraph_list, component_2_subgraphs, CLIST, GcLIST
        
    elif args.task == 'node_reg':
        args.num_features, candidate, C_list, Gc_list, subgraph_list, component_2_subgraphs, CLIST, GcLIST = coarsening_regression(args, dataset[0], 1-args.coarsening_ratio, args.coarsening_method)
        
    elif args.task == 'graph_cls':
        new_dataset = []
        classes = set()
        for i in tqdm(range(len(dataset))):
            try:
                args.num_features, candidate, C_list, Gc_list, subgraph_list, component_2_subgraphs, CLIST, GcLIST = coarsening_classification(args, dataset[i], 1-args.coarsening_ratio, args.coarsening_method)
                Gc = load_graph_data(dataset[i], CLIST, GcLIST, candidate)
                Gs = subgraph_list
                new_dataset.append((dataset[i], Gc, Gs))
                classes.add(dataset[i].y.item())
            except:
                pass
        args.num_classes = len(classes)
        
    else:
        new_dataset = []
        for i in tqdm(range(len(dataset))):
            try:
                args.num_features, candidate, C_list, Gc_list, subgraph_list, component_2_subgraphs, CLIST, GcLIST = coarsening_regression(args, dataset[i], 1-args.coarsening_ratio, args.coarsening_method)
                Gc = load_graph_data(dataset[i], CLIST, GcLIST, candidate)
                Gs = subgraph_list
                new_dataset.append((dataset[i], Gc, Gs))
            except:
                pass

In [71]:
args

Namespace(dataset='cora', experiment='fixed', runs=20, exp_setup='Gc_train_2_Gs_infer', hidden=512, epochs1=100, epochs2=300, num_layers1=2, num_layers2=2, batch_size=128, train_ratio=0.3, val_ratio=0.2, early_stopping=10, extra_node=False, cluster_node=False, super_graph=False, lr=0.01, weight_decay=0.0005, normalize_features=True, coarsening_ratio=0.1, coarsening_method='variation_neighborhoods', task='node_cls', seed=None, multi_prop=False, property=0)

In [72]:
num_features, candidate, C_list, Gc_list, subgraph_list, component_2_subgraphs, CLIST, GcLIST = save_graph(dataset, args, './dataset')

/hdfs1/Data/weather/CoarseGNN_Hrriday/CoPart-GNN/graph_coarsening/coarsening_utils.py:83: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  lk, Uk = sp.sparse.linalg.eigsh(T.toarray(), k=K, which="LM", tol=1e-5)
/hdfs1/Data/weather/CoarseGNN_Hrriday/CoPart-GNN/graph_coarsening/coarsening_utils.py:97: RuntimeWarning: invalid value encountered in power
  dinvsqrt = d ** (-1 / 2)


In [73]:
len(candidate), len(C_list), len(Gc_list), len(subgraph_list), len(component_2_subgraphs), len(CLIST), len(GcLIST)

(78, 2, 2, 409, 78, 78, 78)

In [84]:
ind = 0
Gc_list[ind].N, len(component_2_subgraphs[ind])

(250, 250)

In [103]:
candidate

In [88]:
import torch

# Assume `coarsened_graphs` is a list of PyTorch Geometric Data objects (the coarsened graphs)
# Example: coarsened_graphs = [graph1, graph2, ...]

# Save all coarsened graphs to a .pt file
torch.save(subgraph_list, './dataset/cora/saved/variation_neighborhoods/subgraphs_0.1.pt')

# To load the graphs back, simply use torch.load
loaded_coarsened_graphs = torch.load('./dataset/cora/saved/variation_neighborhoods/subgraphs_0.1.pt')

# Now `loaded_coarsened_graphs` will contain the list of coarsened graphs, similar to `coarsened_graphs`


In [93]:
import pickle

# Assume you have a list of pygsp graphs
# Example: pygsp_graphs = [graph1, graph2, ...]

# Save the list of pygsp graphs to a file
with open('./dataset/cora/saved/variation_neighborhoods/coarsened_0.1.pkl', 'wb') as f:
    pickle.dump(Gc_list, f)

# To load the graphs back, use pickle.load
with open('./dataset/cora/saved/variation_neighborhoods/coarsened_0.1.pkl', 'rb') as f:
    loaded_pygsp_graphs = pickle.load(f)

# `loaded_pygsp_graphs` will now contain the list of pygsp graphs, similar to `pygsp_graphs`


In [98]:
import pickle

# Assume C_list is your list of sparse matrices
# Example: C_list = [matrix1, matrix2, ...]

# Save the list of sparse matrices to a file
with open('./dataset/cora/saved/variation_neighborhoods/clist_0.1.pkl', 'wb') as f:
    pickle.dump(C_list, f)

# To load the matrices back, use pickle.load
with open('./dataset/cora/saved/variation_neighborhoods/clist_0.1.pkl', 'rb') as f:
    loaded_C_list = pickle.load(f)

# `loaded_C_list` will contain the list of sparse matrices, similar to `C_list`


In [101]:
import os

In [102]:
path = './dataset/cora/saved/variation_neighborhoods/'
if not os.path.exists(path):
    os.makedirs(path)

In [107]:
enz_subgraph = torch.load('./dataset/ENZYMES/saved/algebraic_JC/0.9c_subgraph_list.pt')
with open('./dataset/ENZYMES/saved/algebraic_JC/0.9c_Gc_list.pkl', 'rb') as f:
    enz_Gc_list = pickle.load(f)

In [113]:
ind = 100
len(enz_subgraph[ind]), enz_Gc_list[ind]

(41, Data(x=[41, 3], edge_index=[2, 154], y=[1]))

NEW

In [9]:
import pickle
import torch
from torch_geometric.datasets import TUDataset

In [10]:
saved_graph_list = pickle.load(open("./dataset/AIDS/saved/algebraic_JC/0.7c_saved_graph_list.pkl", "rb"))
subgraph_list = torch.load("./dataset/AIDS/saved/algebraic_JC/0.7c_subgraph_list.pt")
Gc_list = pickle.load(open("./dataset/AIDS/saved/algebraic_JC/0.7c_Gc_list.pkl", "rb"))

In [12]:
len(saved_graph_list), len(subgraph_list), len(Gc_list)

(2000, 0, 0)

In [6]:
dataset = TUDataset(root='./dataset', name='AIDS')

In [13]:
dataset

AIDS(2000)